# LinguaBridge AI — T5-small Translation Web App
A Gradio web app using Hugging Face `google-t5/t5-small`, adapted from the supplied summarization notebook structure.

**Important:** `t5-small` is a general-purpose base checkpoint, not a translation-fine-tuned multilingual model. The translation prompt is a best-effort demonstration and may not produce reliable translations for all language pairs.

## Step 1 — Install dependencies
Run this cell in Jupyter or Google Colab. On a local machine, you can alternatively run `pip install -r requirements.txt` in the project folder.

In [1]:
%pip install -q transformers gradio sentencepiece torch

## Step 2 — Load the Hugging Face model
The first run downloads the model files and requires an internet connection.

In [2]:
import warnings
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging

warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL_NAME = "google-t5/t5-small"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()
print("Model loaded.")

Loading google-t5/t5-small...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded.


## Step 3 — Create the translation function

In [3]:
LANGUAGES = [
    "English", "French", "German", "Spanish", "Italian", "Portuguese",
    "Dutch", "Romanian", "Russian", "Arabic", "Hindi", "Bengali",
    "Tamil", "Telugu", "Japanese", "Chinese"
]

def translate_text(text, source_language, target_language, max_tokens=128):
    if not text or not text.strip():
        return "Please enter some text to translate."
    if source_language == target_language:
        return "Choose two different languages for translation."

    # Best-effort T5 task prompt; this base model is not translation-fine-tuned.
    task_prompt = f"translate {source_language} to {target_language}: {text.strip()}"
    inputs = tokenizer(task_prompt, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            num_beams=4,
            early_stopping=True
        )
    result = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    if not result:
        return "The model did not return a translation. Try a shorter or simpler sentence."
    return result

def translate_ui(text, source_language, target_language, max_tokens):
    try:
        return translate_text(text, source_language, target_language, max_tokens)
    except Exception as exc:
        return f"Translation failed: {type(exc).__name__}: {exc}"

## Step 4 — Build the Gradio web interface

In [5]:
CUSTOM_CSS = """
.gradio-container {max-width: 1050px !important; margin: auto !important;}
#hero {text-align:center; padding: 18px 10px 8px;}
#hero h1 {font-size: 2.6rem; margin-bottom: 0.25rem;}
#hero p {font-size: 1.05rem; opacity: .82;}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=CUSTOM_CSS, title='LinguaBridge AI') as demo:
    gr.HTML("""<div id="hero"><h1>🌐 LinguaBridge AI</h1><p>Explore language translation with Hugging Face's T5-small model.</p></div>""")
    with gr.Row():
        source = gr.Dropdown(choices=LANGUAGES, value='English', label='Translate from')
        target = gr.Dropdown(choices=LANGUAGES, value='French', label='Translate to')
    source_text = gr.Textbox(label='Your text', placeholder='Type or paste text here...', lines=7)
    max_tokens = gr.Slider(minimum=32, maximum=256, value=128, step=16, label='Maximum output tokens')
    with gr.Row():
        translate_btn = gr.Button('Translate', variant='primary')
        clear_btn = gr.Button('Clear')
    output_text = gr.Textbox(label='Translation', lines=7, interactive=False)
    translate_btn.click(translate_ui, inputs=[source_text, source, target, max_tokens], outputs=output_text)
    source_text.submit(translate_ui, inputs=[source_text, source, target, max_tokens], outputs=output_text)
    clear_btn.click(lambda: ('', ''), inputs=[], outputs=[source_text, output_text])
    gr.Markdown('''**Model note:** `google-t5/t5-small` is a base checkpoint, not a dedicated multilingual translation model. Results may be inaccurate or remain in the source language. Do not use this demo for high-stakes translations.''')

gr.HTML("""<div id="hero"><h1>🌐 LinguaBridge AI</h1><p>Explore language translation with Hugging Face's T5-small model.</p></div>""")

## Step 5 — Launch the web app

In [6]:
# Run this cell to start the local Gradio app.
# Open the local URL shown in the output, usually http://127.0.0.1:7860
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://22b62a507c34a68a82.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Notes
- The first launch downloads the model from Hugging Face.
- CPU inference is supported but may be slower.
- For dependable multilingual translation, replace the base checkpoint with a translation-fine-tuned model and follow that model's documented prompting format.